# 面试题：关键词检索应该怎样设计，BM25 又怎样从零实现？

## 可以直接复述的回答

关键词检索首先要把文档切成可检索字段，再建立 term 到文档的倒排统计；BM25 只是排序函数，不替代分词、过滤、召回和评估。它用 IDF 奖励稀有词，用饱和 TF 抑制关键词堆砌，并用文档长度项避免长文天然占优。实现时必须能解释每个 query-term 对每篇文档的贡献，而不是只返回一个总分。查询里的重复词也不能无条件重复加分，否则“退款退款退款”会把垃圾页面推到前面。本题用 9 篇客服知识库文章与 6 个真实意图，对比原始词频基线并展开 TF、IDF、长度归一项。最后复现重复 query token 造成的排序攻击，再用去重后的 BM25 修正。

## 真实案例

样本模拟客服帮助中心的标题和人工分词字段，其中包含一篇故意堆砌“退款”的低质量页面。数据为离线教学构造，不含用户信息，也不能代表真实中文分词器或线上点击分布。

In [1]:
import math  # 导入对数函数以计算 BM25 的 IDF
from collections import Counter  # 导入计数器以统计词频和文档频率
from pprint import pprint  # 导入结构化打印函数以展示分项账本
documents = [{"id": "D1", "标题": "退款到账时限", "tokens": ["退款", "到账", "时限", "原路", "退回", "三个", "工作日"]}, {"id": "D2", "标题": "如何提交退款申请", "tokens": ["订单", "退款", "申请", "路径", "售后", "提交"]}, {"id": "D3", "标题": "电子发票下载", "tokens": ["电子", "发票", "下载", "邮箱", "订单"]}, {"id": "D4", "标题": "忘记密码处理", "tokens": ["忘记", "密码", "重置", "登录", "账号"]}, {"id": "D5", "标题": "物流进度查询", "tokens": ["物流", "快递", "进度", "查询", "订单"]}, {"id": "D6", "标题": "取消未发货订单", "tokens": ["取消", "订单", "未发货", "支付", "关闭"]}, {"id": "D7", "标题": "退款凭证要求", "tokens": ["退款", "凭证", "照片", "破损", "售后"]}, {"id": "D8", "标题": "修改收货地址", "tokens": ["修改", "收货", "地址", "订单", "发货"]}, {"id": "D9", "标题": "低质量退款聚合页", "tokens": ["退款", "退款", "退款", "退款", "退款", "退款", "退款", "退款", "退款", "退款", "广告", "优惠"]}]  # 构造九篇有字段语义的知识库文档
queries = [{"id": "Q1", "问题": "退款多久能到账", "tokens": ["退款", "到账", "时限"], "相关": "D1"}, {"id": "Q2", "问题": "怎么申请退款", "tokens": ["申请", "退款", "路径"], "相关": "D2"}, {"id": "Q3", "问题": "退款要什么证明", "tokens": ["退款", "凭证", "照片"], "相关": "D7"}, {"id": "Q4", "问题": "电子发票怎么下载", "tokens": ["电子", "发票", "下载"], "相关": "D3"}, {"id": "Q5", "问题": "忘记密码怎么办", "tokens": ["忘记", "密码", "重置"], "相关": "D4"}, {"id": "Q6", "问题": "未发货如何撤单", "tokens": ["取消", "订单", "未发货"], "相关": "D6"}]  # 构造六个带人工相关文档的查询
print("知识库输入预览：")  # 输出真实案例标题
pprint([{"id": doc["id"], "标题": doc["标题"], "长度": len(doc["tokens"]), "tokens": doc["tokens"]} for doc in documents])  # 展示文档字段、长度与分词
print("查询与人工相关性：")  # 输出评估标签标题
pprint(queries)  # 展示六条查询及唯一相关文档

知识库输入预览：
[{'id': 'D1',
  'tokens': ['退款', '到账', '时限', '原路', '退回', '三个', '工作日'],
  '标题': '退款到账时限',
  '长度': 7},
 {'id': 'D2',
  'tokens': ['订单', '退款', '申请', '路径', '售后', '提交'],
  '标题': '如何提交退款申请',
  '长度': 6},
 {'id': 'D3',
  'tokens': ['电子', '发票', '下载', '邮箱', '订单'],
  '标题': '电子发票下载',
  '长度': 5},
 {'id': 'D4',
  'tokens': ['忘记', '密码', '重置', '登录', '账号'],
  '标题': '忘记密码处理',
  '长度': 5},
 {'id': 'D5',
  'tokens': ['物流', '快递', '进度', '查询', '订单'],
  '标题': '物流进度查询',
  '长度': 5},
 {'id': 'D6',
  'tokens': ['取消', '订单', '未发货', '支付', '关闭'],
  '标题': '取消未发货订单',
  '长度': 5},
 {'id': 'D7',
  'tokens': ['退款', '凭证', '照片', '破损', '售后'],
  '标题': '退款凭证要求',
  '长度': 5},
 {'id': 'D8',
  'tokens': ['修改', '收货', '地址', '订单', '发货'],
  '标题': '修改收货地址',
  '长度': 5},
 {'id': 'D9',
  'tokens': ['退款',
             '退款',
             '退款',
             '退款',
             '退款',
             '退款',
             '退款',
             '退款',
             '退款',
             '退款',
             '广告',
             '优惠'],
  '标题': '低质量退款聚合页',
 

## Baseline / 基线：直接累加原始词频

最简单的关键词方案把 query 中每个词在文档里的出现次数相加。它能工作，但会被“退款”重复十次的页面轻易操纵，也完全不知道“到账”比“订单”更稀有。

In [2]:
def raw_tf_score(query_tokens, document):  # 定义不做饱和与长度归一的原始词频基线
    frequencies = Counter(document["tokens"])  # 统计当前文档的词频
    return sum(frequencies[token] for token in query_tokens)  # 直接累加查询词出现次数
def rank_with_score(score_function, query_tokens):  # 定义统一排序辅助函数
    scored = [{"doc_id": doc["id"], "标题": doc["标题"], "score": score_function(query_tokens, doc)} for doc in documents]  # 计算所有文档分数
    return sorted(scored, key=lambda row: (-row["score"], row["doc_id"]))  # 按分数降序和文档编号稳定排序
baseline_rankings = {query["id"]: rank_with_score(raw_tf_score, query["tokens"]) for query in queries}  # 执行六条查询的基线排序
baseline_hits = sum(baseline_rankings[query["id"]][0]["doc_id"] == query["相关"] for query in queries)  # 统计基线 Top1 命中数
print("原始 TF Baseline 的逐查询 Top3：")  # 输出基线排名标题
pprint([{"查询": query["问题"], "期望": query["相关"], "Top3": [(row["doc_id"], row["score"]) for row in baseline_rankings[query["id"]][:3]]} for query in queries])  # 展示同数据基线排名
print(f"Baseline Top1 命中：{baseline_hits}/{len(queries)}")  # 输出基线聚合指标

原始 TF Baseline 的逐查询 Top3：
[{'Top3': [('D9', 10), ('D1', 3), ('D2', 1)], '期望': 'D1', '查询': '退款多久能到账'},
 {'Top3': [('D9', 10), ('D2', 3), ('D1', 1)], '期望': 'D2', '查询': '怎么申请退款'},
 {'Top3': [('D9', 10), ('D7', 3), ('D1', 1)], '期望': 'D7', '查询': '退款要什么证明'},
 {'Top3': [('D3', 3), ('D1', 0), ('D2', 0)], '期望': 'D3', '查询': '电子发票怎么下载'},
 {'Top3': [('D4', 3), ('D1', 0), ('D2', 0)], '期望': 'D4', '查询': '忘记密码怎么办'},
 {'Top3': [('D6', 3), ('D2', 1), ('D3', 1)], '期望': 'D6', '查询': '未发货如何撤单'}]
Baseline Top1 命中：3/6


## 手写 BM25：DF、IDF、TF 饱和与长度归一

这里采用常见公式 IDF=log(1+(N-df+0.5)/(df+0.5))。长度项为 k1×(1-b+b×dl/avgdl)，它与 TF 一起决定饱和权重；下面会把这些数逐项打印出来。

In [3]:
document_count = len(documents)  # 记录语料文档总数 N
average_length = sum(len(doc["tokens"]) for doc in documents) / document_count  # 计算平均文档长度 avgdl
document_frequency = Counter()  # 创建文档频率统计器
for document in documents:  # 遍历每篇知识库文档
    for token in set(document["tokens"]):  # 同一词在一篇文档内只计一次 DF
        document_frequency[token] += 1  # 累加包含该词的文档数
def inverse_document_frequency(token):  # 定义 Robertson 风格的正值 IDF
    frequency = document_frequency.get(token, 0)  # 读取当前词的文档频率
    return math.log(1 + (document_count - frequency + 0.5) / (frequency + 0.5))  # 计算平滑后的 IDF
print(f"语料统计：N={document_count}, avgdl={average_length:.3f}")  # 输出长度归一所需全局量
pprint({token: {"df": document_frequency[token], "idf": round(inverse_document_frequency(token), 4)} for token in ["退款", "到账", "订单", "凭证", "不存在词"]})  # 展示高频词、稀有词与 OOV 词的统计

语料统计：N=9, avgdl=6.111
{'不存在词': {'df': 0, 'idf': 2.9957},
 '凭证': {'df': 1, 'idf': 1.8971},
 '到账': {'df': 1, 'idf': 1.8971},
 '订单': {'df': 5, 'idf': 0.5978},
 '退款': {'df': 4, 'idf': 0.7985}}


In [4]:
def bm25_score(query_tokens, document, k1=1.5, b=0.75):  # 定义可解释的 BM25 打分函数
    frequencies = Counter(document["tokens"])  # 统计当前文档的词频 TF
    document_length = len(document["tokens"])  # 读取当前文档长度 dl
    length_term = k1 * (1 - b + b * document_length / average_length)  # 计算长度归一项
    total_score = 0.0  # 初始化文档总分
    contribution_ledger = []  # 收集每个查询词的分项账本
    for token in dict.fromkeys(query_tokens):  # 对查询词去重以避免重复输入攻击
        term_frequency = frequencies[token]  # 读取该词在当前文档的 TF
        idf = inverse_document_frequency(token)  # 计算该词的 IDF
        tf_weight = term_frequency * (k1 + 1) / (term_frequency + length_term) if term_frequency else 0.0  # 计算带饱和与长度归一的 TF 权重
        contribution = idf * tf_weight  # 计算该词对总分的贡献
        total_score += contribution  # 累加当前词贡献
        contribution_ledger.append({"term": token, "TF": term_frequency, "IDF": round(idf, 4), "dl": document_length, "长度项": round(length_term, 4), "TF权重": round(tf_weight, 4), "贡献": round(contribution, 4)})  # 保存完整可解释中间量
    return total_score, contribution_ledger  # 同时返回总分和分项账本
def bm25_total(query_tokens, document):  # 定义适配统一排序接口的 BM25 总分函数
    total_score, _ = bm25_score(query_tokens, document)  # 调用核心函数并忽略账本
    return total_score  # 返回当前文档总分
bm25_rankings = {query["id"]: rank_with_score(bm25_total, query["tokens"]) for query in queries}  # 对六条查询执行 BM25 排序
bm25_hits = sum(bm25_rankings[query["id"]][0]["doc_id"] == query["相关"] for query in queries)  # 统计 BM25 Top1 命中数
example_query = queries[0]  # 选择到账时限问题展开中间量
detail_rows = []  # 创建 TF、IDF 与长度项明细表
for document in documents:  # 遍历所有候选文档
    total_score, ledger = bm25_score(example_query["tokens"], document)  # 计算当前文档总分及逐词贡献
    detail_rows.append({"doc_id": document["id"], "标题": document["标题"], "总分": round(total_score, 4), "逐词分项": ledger})  # 保存当前文档的完整解释
print("Q1 的 TF / IDF / 文档长度项 / 贡献账本：")  # 输出机制解释标题
pprint(detail_rows)  # 展示每篇文档对每个 query-term 的具体贡献

Q1 的 TF / IDF / 文档长度项 / 贡献账本：
[{'doc_id': 'D1',
  '总分': 4.3106,
  '标题': '退款到账时限',
  '逐词分项': [{'IDF': 0.7985,
            'TF': 1,
            'TF权重': 0.9386,
            'dl': 7,
            'term': '退款',
            '贡献': 0.7495,
            '长度项': 1.6636},
           {'IDF': 1.8971,
            'TF': 1,
            'TF权重': 0.9386,
            'dl': 7,
            'term': '到账',
            '贡献': 1.7806,
            '长度项': 1.6636},
           {'IDF': 1.8971,
            'TF': 1,
            'TF权重': 0.9386,
            'dl': 7,
            'term': '时限',
            '贡献': 1.7806,
            '长度项': 1.6636}]},
 {'doc_id': 'D2',
  '总分': 0.8051,
  '标题': '如何提交退款申请',
  '逐词分项': [{'IDF': 0.7985,
            'TF': 1,
            'TF权重': 1.0082,
            'dl': 6,
            'term': '退款',
            '贡献': 0.8051,
            '长度项': 1.4795},
           {'IDF': 1.8971,
            'TF': 0,
            'TF权重': 0.0,
            'dl': 6,
            'term': '到账',
            '贡献': 0.0,
           

## 结果表与结果解读

原始 TF 会让堆砌十次“退款”的 D9 抢走多个查询的第一名。BM25 对 TF 做饱和，并让“到账、时限、凭证、照片”等稀有词贡献更大，因此能把真正回答问题的文章排到前面。注意这里的 Top1 命中来自小型人工判断，不是线上 CTR 结论。

In [5]:
result_rows = []  # 创建同数据逐查询对照表
for query in queries:  # 遍历六条带相关性标签的查询
    baseline_top = baseline_rankings[query["id"]][0]  # 读取基线第一名
    bm25_top = bm25_rankings[query["id"]][0]  # 读取 BM25 第一名
    result_rows.append({"问题": query["问题"], "期望": query["相关"], "原始TF Top1": baseline_top["doc_id"], "原始TF分": round(baseline_top["score"], 4), "BM25 Top1": bm25_top["doc_id"], "BM25分": round(bm25_top["score"], 4), "是否修正": baseline_top["doc_id"] != bm25_top["doc_id"]})  # 保存逐样本可比较结果
print("逐查询结果对照：")  # 输出结果表标题
pprint(result_rows)  # 展示基线与 BM25 的每条决策
print(f"Top1 命中从 {baseline_hits}/{len(queries)} 提升到 {bm25_hits}/{len(queries)}")  # 输出同一指标下的汇总变化

逐查询结果对照：
[{'BM25 Top1': 'D1',
  'BM25分': 4.3106,
  '原始TF Top1': 'D9',
  '原始TF分': 10,
  '是否修正': True,
  '期望': 'D1',
  '问题': '退款多久能到账'},
 {'BM25 Top1': 'D2',
  'BM25分': 4.6306,
  '原始TF Top1': 'D9',
  '原始TF分': 10,
  '是否修正': True,
  '期望': 'D2',
  '问题': '怎么申请退款'},
 {'BM25 Top1': 'D7',
  'BM25分': 5.002,
  '原始TF Top1': 'D9',
  '原始TF分': 10,
  '是否修正': True,
  '期望': 'D7',
  '问题': '退款要什么证明'},
 {'BM25 Top1': 'D3',
  'BM25分': 6.1985,
  '原始TF Top1': 'D3',
  '原始TF分': 3,
  '是否修正': False,
  '期望': 'D3',
  '问题': '电子发票怎么下载'},
 {'BM25 Top1': 'D4',
  'BM25分': 6.1985,
  '原始TF Top1': 'D4',
  '原始TF分': 3,
  '是否修正': False,
  '期望': 'D4',
  '问题': '忘记密码怎么办'},
 {'BM25 Top1': 'D6',
  'BM25分': 4.7835,
  '原始TF Top1': 'D6',
  '原始TF分': 3,
  '是否修正': False,
  '期望': 'D6',
  '问题': '未发货如何撤单'}]
Top1 命中从 3/6 提升到 6/6


## 失败案例：重复 query token 被重复加分

如果实现直接遍历 query 列表，“退款”重复五次就会被当成五份独立证据，关键词堆砌页再次登顶。修正可以是像本实现一样对 query term 去重；需要保留 query term frequency 时，则应另设有上限的 qtf 因子。

In [6]:
def unsafe_bm25_score(query_tokens, document):  # 定义未处理重复查询词的错误实现
    frequencies = Counter(document["tokens"])  # 统计当前文档词频
    length_term = 1.5 * (1 - 0.75 + 0.75 * len(document["tokens"]) / average_length)  # 计算与主实现相同的长度项
    total_score = 0.0  # 初始化错误实现的总分
    for token in query_tokens:  # 错误地逐个消费包含重复项的查询列表
        term_frequency = frequencies[token]  # 读取当前词的 TF
        tf_weight = term_frequency * 2.5 / (term_frequency + length_term) if term_frequency else 0.0  # 计算当前词的 TF 权重
        total_score += inverse_document_frequency(token) * tf_weight  # 重复词会被多次累加
    return total_score  # 返回受重复 token 影响的总分
attacked_query = ["退款", "退款", "退款", "退款", "退款", "到账", "时限"]  # 构造重复关键词攻击查询
unsafe_top = rank_with_score(unsafe_bm25_score, attacked_query)[0]  # 获取错误实现的第一名
safe_top = rank_with_score(bm25_total, attacked_query)[0]  # 获取去重 BM25 的第一名
print("失败案例的第一名：", unsafe_top)  # 展示重复词让低质量页面登顶
print("修正后的第一名：", safe_top)  # 展示 query 去重后的正确文档

失败案例的第一名： {'doc_id': 'D9', '标题': '低质量退款聚合页', 'score': 7.931718131113859}
修正后的第一名： {'doc_id': 'D1', '标题': '退款到账时限', 'score': 4.31059934521202}


## 生产差距

线上系统还需要中文分词与领域词典、标题/正文/标签字段权重、停用词、同义词、ACL 与时间过滤、倒排表压缩和增量 IDF。参数 k1、b 应通过判断集调优，相关性要监控分群指标与零结果率；搜索日志还需隐私治理和反作弊，不能只靠这个内存版排序器。

In [7]:
assert len(documents) == 9  # 验证案例包含正常文档与关键词堆砌反例
assert baseline_hits < bm25_hits  # 验证同数据上的 BM25 优于原始词频基线
assert bm25_hits == len(queries)  # 验证六条教学查询的正确文档均排在第一名
assert unsafe_top["doc_id"] == "D9"  # 验证重复查询词能攻击错误实现
assert safe_top["doc_id"] == "D1"  # 验证去重后的 BM25 恢复正确到账文档
assert inverse_document_frequency("到账") > inverse_document_frequency("订单")  # 验证稀有词获得更高 IDF
print("最小回归测试通过：BM25 分项、排序改进与重复词门禁均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：BM25 分项、排序改进与重复词门禁均满足预期
